In [ ]:
import requests
import json
import time
import notebookutils
from notebookutils import mssparkutils
from pyspark.sql import Row
from pyspark.sql.functions import col, max as spark_max
from pyspark.sql.types import StructType, StructField, TimestampType
from datetime import datetime, timedelta, timezone


In [ ]:
kusto_cluster_uri = 'https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com'
tenant_id = "9e929790-272d-4977-a2ab-301443c11ece"
client_id = "b5c04c9c-0588-418f-8f60-2d83d38cb635"
azure_key_vault_name = "ab-fabric-automation-akv"
secret_name = "fabric-monit-secret"

In [ ]:
azure_key_vault_url=f"https://{azure_key_vault_name}.vault.azure.net"
client_secret = notebookutils.credentials.getSecret(azure_key_vault_url,secret_name)

In [ ]:
# Capture Start Time (for watermark)
execution_start_time = datetime.now(timezone.utc)
print(f'Execution Start Time: {execution_start_time}')

# --- Configuration ---
# Kusto / Eventhouse
kusto_database    = 'MonitoringEventhouse'

# Fabric REST base
fabric_api_base   = 'https://api.fabric.microsoft.com/v1'

# Lakehouse Delta table that holds the high-watermark for AlertLogs processing.
# Replaces the previous Kusto DeleteLastRunTime table.
delete_config_table = 'Delete_Items_Config'

# Auth: tokens are obtained from the notebook's run identity via
# notebookutils.credentials.getToken (see next cell). No SPN secret needed.


In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type":   "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         scope,
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve token for scope {scope}: {e}")
        return None

def get_tokens():
    global auth_token, kusto_token
    kusto_scope = f"{kusto_cluster_uri}/default"
    auth_scope = "https://analysis.windows.net/powerbi/api/.default"
    auth_token = get_spn_token(auth_scope)
    kusto_token = get_spn_token(kusto_scope)
    if not kusto_token or not auth_token:
        raise Exception("Error: one or more tokens missing.")

get_tokens()

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 87, Finished, Available, Finished, False)

In [ ]:
# Create / verify the Lakehouse watermark table.
# One row holds the upper-bound timestamp of the most recent successful run.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {delete_config_table} (
        LastRunTime TIMESTAMP,
        UpdatedAt   TIMESTAMP
    )
    USING DELTA
""")

print(f"Verified configuration table: {delete_config_table}")


def load_watermark():
    """Return the last-processed upper-bound (UTC) from the Lakehouse config table, or None."""
    df_config = spark.table(delete_config_table)
    max_row = df_config.agg(spark_max("LastRunTime").alias("max_ts")).collect()
    last_run = max_row[0]["max_ts"]
    if last_run is None:
        print("No watermark in Delete_Items_Config — will process from default lookback.")
        return None
    if last_run.tzinfo is None:
        last_run = last_run.replace(tzinfo=timezone.utc)
    print(f"Watermark loaded: {last_run}")
    return last_run


In [ ]:
# --- Item Type Mappings ---

# Maps item type → REST API endpoint segment
ITEM_ENDPOINT_MAP = {
    "ApacheAirflowJob":              "ApacheAirflowJobs",
    "CopyJob":                       "copyJobs",
    "CosmosDBDatabase":              "cosmosDbDatabases",
    "Dashboard":                     "dashboards",
    "DataAgent":                     "dataAgents",
    "DataFlow":                      "dataflows",
    "DataPipeline":                  "dataPipelines",
    "Environment":                   "environments",
    "Eventhouse":                    "eventhouses",
    "EventSchemaSet":                "eventSchemaSets",
    "GraphQLApi":                    "GraphQLApis",
    "KQLDashboard":                  "kqlDashboards",
    "KQLQueryset":                   "kqlQuerysets",
    "Lakehouse":                     "lakehouses",
    "Map":                           "maps",
    "MirroredAzureDatabricksCatalog": "mirroredAzureDatabricksCatalogs",
    "MirroredDatabase":              "mirroredDatabases",
    "MountedDataFactory":            "mountedDataFactories",
    "Notebook":                      "notebooks",
    "PaginatedReport":               "reports",
    "Reflex":                        "reflexes",
    "Report":                        "reports",
    "SemanticModel":                 "semanticModels",
    "SparkJobDefinition":            "sparkJobDefinitions",
    "SQLDatabase":                   "sqlDatabases",
    "UserDataFunction":              "UserDataFunctions",
    "VariableLibrary":               "VariableLibraries",
    "Warehouse":                     "warehouses",
}


# --- Define Helper Functions ---

def delete_fabric_item(workspace_id, item_id, item_kind, max_retries=2):
    """
    Deletes a specific item from a Fabric Workspace.
    API: DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/{endpoint}/{itemId}
    Returns: (status_message, error_code, description)
    """
    if not auth_token:
        print("Skipping Delete: No Fabric token available.")
        return "DeleteFailed", "NoToken", "No Fabric token available"

    # Use specific endpoint if mapped, otherwise default to generic 'items'
    endpoint = ITEM_ENDPOINT_MAP.get(item_kind, "items")
    url = f"{fabric_api_base}/workspaces/{workspace_id}/{endpoint}/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json"
    }
    
    last_exception = None
    
    for attempt in range(max_retries):
        try:
            response = requests.delete(url, headers=headers)
            
            # 200 OK or 204 No Content are typical success codes for DELETE
            if response.status_code in [200, 204]:
                return "DeletedItem", str(response.status_code), "Success"
            elif response.status_code == 404:
                print(f"SKIPPED: Item {item_id} not found (already deleted?).")
                return "404: File Not Found", "404", "Item Not Found"
            elif response.status_code == 401:
                # Unauthorized - Need to add permissions
                print(f"UNAUTHORIZED (401): No permission for item {item_id} in ws {workspace_id}.")
                return "401: Unauthorized", "401", response.text
            elif response.status_code == 429:
                # Handle Rate Limiting
                print(f"RATE LIMIT (429) Headers: {response.headers}")
                retry_after = int(response.headers.get("Retry-After", 60))
                print(f"RATE LIMIT (429): Waiting {retry_after} seconds before retry {attempt + 1}/{max_retries}...")
                time.sleep(retry_after)
                continue
            else:
                print(f"FAILED: Could not delete item {item_id}. Status: {response.status_code}, Response: {response.text}")
                return "DeleteFailed", str(response.status_code), response.text
                
        except Exception as e:
            print(f"EXCEPTION: Error deleting item {item_id}: {e}")
            last_exception = e
    
    print(f"FAILED: Max retries ({max_retries}) exceeded for item {item_id}.")
    desc = str(last_exception) if last_exception else "Max retries exceeded"
    return "DeleteFailed", "MaxRetries", desc

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 88, Finished, Available, Finished, False)

In [ ]:
KUSTO_MAX_RETRIES = 5
KUSTO_RETRY_BACKOFF_SECONDS = 5


def get_alerts_to_process(upper_bound, lower_bound=None):
    print("Reading from Kusto query...")

    upper_bound_str = upper_bound.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    # Resolve the lower bound:
    #   - explicit arg (loop iteration), else
    #   - watermark from the Lakehouse config table, else
    #   - 100-day lookback (first ever run).
    if lower_bound is None:
        lower_bound = load_watermark()
    if lower_bound is None:
        lower_bound = upper_bound - timedelta(days=100)

    lower_bound_str = lower_bound.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    kusto_query = f"""
    AlertLogs
    | where ingestion_time() >= datetime('{lower_bound_str}')
    | where ingestion_time() < datetime('{upper_bound_str}')
    | summarize arg_max(ingestion_time(), WorkspaceName, wstime, data_itemName, AlertStatus, data_itemKind) by WorkspaceId, data_itemId
    | where AlertStatus in ("EmailSent", "NoEmail")
    | project WorkspaceName, WorkspaceId, wstime, data_itemKind, data_itemName, data_itemId
    """

    attempt = 0
    while True:
        attempt += 1
        try:
            df_alerts = spark.read \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("kustoCluster", kusto_cluster_uri) \
                .option("kustoDatabase", kusto_database) \
                .option("kustoQuery", kusto_query) \
                .option("accessToken", kusto_token) \
                .load()

            alerts_list = df_alerts.select(
                "WorkspaceId", "WorkspaceName", "wstime",
                "data_itemKind", "data_itemName", "data_itemId"
            ).collect()

            print(f"Found {len(alerts_list)} alerts to process (window: {lower_bound_str} -> {upper_bound_str}).")
            return alerts_list

        except Exception as e:
            print(f"Kusto read failed (attempt {attempt}/{KUSTO_MAX_RETRIES}): {e}")
            if attempt >= KUSTO_MAX_RETRIES:
                raise RuntimeError(
                    f"Failed to read AlertLogs from Kusto after {KUSTO_MAX_RETRIES} attempts. "
                    f"Aborting notebook."
                ) from e
            time.sleep(KUSTO_RETRY_BACKOFF_SECONDS * attempt)


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 89, Finished, Available, Finished, False)

In [ ]:
def process_alerts(alerts_list):
    results_to_log    = []
    exceptions_to_log = []
    deferred_deletion_list = []

    if not alerts_list:
        print("No alerts to process.")
        return results_to_log, exceptions_to_log

    if not auth_token:
        raise Exception("Aborting Deletion Process: No Fabric token available.")
    if not kusto_token:
        raise Exception("Aborting Deletion Process: No Kusto token available.")

    print(f"Starting deletion loop for {len(alerts_list)} items...")

    for row in alerts_list:
        item_id   = row['data_itemId']
        ws_id     = row['WorkspaceId']
        ws_name   = row['WorkspaceName']
        item_kind = row['data_itemKind']

        print(f"Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}] in ws: {ws_name} ({ws_id})")
        status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)

        # Defer items that return 400 UnknownError (possible dependency issue)
        if code == "400" and "UnknownError" in desc:
            print(f"DEFERRING: Item {item_id} returned 400 UnknownError (possible dependency). Saving for retry.")
            deferred_deletion_list.append(row)
            continue

        if status == "DeletedItem" or "404" in status:
            results_to_log.append({
                "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                "wstime": row['wstime'], "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'], "data_itemId": item_id,
                "AlertStatus": status,
            })
        else:
            exceptions_to_log.append({
                "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                "wstime": row['wstime'], "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'], "data_itemId": item_id,
                "AlertStatus": status, "ErrorCode": code, "Description": desc,
            })

    # Retry deferred items after the initial pass
    if deferred_deletion_list:
        print(f"\n--- Retrying {len(deferred_deletion_list)} deferred items after initial pass ---")
        for row in deferred_deletion_list:
            item_id   = row['data_itemId']
            ws_id     = row['WorkspaceId']
            ws_name   = row['WorkspaceName']
            item_kind = row['data_itemKind']

            print(f"RETRY Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}]")
            status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)

            if status == "DeletedItem" or "404" in status:
                results_to_log.append({
                    "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                    "wstime": row['wstime'], "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'], "data_itemId": item_id,
                    "AlertStatus": status,
                })
            else:
                exceptions_to_log.append({
                    "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                    "wstime": row['wstime'], "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'], "data_itemId": item_id,
                    "AlertStatus": status, "ErrorCode": code, "Description": desc,
                })

    print(f"Deletion processing complete. Success/404: {len(results_to_log)}, Exceptions: {len(exceptions_to_log)}")
    return results_to_log, exceptions_to_log


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 90, Finished, Available, Finished, False)

In [ ]:
def _kusto_write_with_retry(df, table_name, label):
    """Write a Spark DataFrame to a Kusto table with retry; raise on final failure."""
    attempt = 0
    while True:
        attempt += 1
        try:
            df.write \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("kustoCluster", kusto_cluster_uri) \
                .option("kustoDatabase", kusto_database) \
                .option("kustoTable", table_name) \
                .option("accessToken", kusto_token) \
                .option("tableCreateOptions", "CreateIfNotExist") \
                .mode("Append") \
                .save()
            print(f"Successfully logged {label} to {table_name}.")
            return
        except Exception as e:
            print(f"Failed to log {label} to {table_name} (attempt {attempt}/{KUSTO_MAX_RETRIES}): {e}")
            if attempt >= KUSTO_MAX_RETRIES:
                raise RuntimeError(
                    f"Failed to write to Kusto table '{table_name}' after {KUSTO_MAX_RETRIES} attempts. "
                    f"Aborting notebook."
                ) from e
            time.sleep(KUSTO_RETRY_BACKOFF_SECONDS * attempt)


def log_results(results_to_log, exceptions_to_log):
    if results_to_log:
        print(f"Logging {len(results_to_log)} entries to AlertLogs...")
        log_df = spark.createDataFrame([Row(**r) for r in results_to_log])
        _kusto_write_with_retry(log_df, "AlertLogs", "alert statuses")
    else:
        print("No standard results to log.")

    if exceptions_to_log:
        print(f"Logging {len(exceptions_to_log)} exceptions to DeleteException...")
        exc_df = spark.createDataFrame([Row(**r) for r in exceptions_to_log])
        _kusto_write_with_retry(exc_df, "DeleteException", "exceptions")
    else:
        print("No exceptions to log.")


def update_watermark(new_watermark):
    """Upsert the high-watermark into the Lakehouse Delta config table, with retry."""
    if new_watermark is None:
        print("No watermark to update.")
        return

    new_watermark_str = new_watermark.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

    attempt = 0
    while True:
        attempt += 1
        try:
            spark.sql(f"""
                MERGE INTO {delete_config_table} AS t
                USING (SELECT CAST('{new_watermark_str}' AS TIMESTAMP) AS LastRunTime) AS s
                ON (1 = 1)
                WHEN MATCHED THEN
                    UPDATE SET t.LastRunTime = s.LastRunTime, t.UpdatedAt = current_timestamp()
                WHEN NOT MATCHED THEN
                    INSERT (LastRunTime, UpdatedAt) VALUES (s.LastRunTime, current_timestamp())
            """)
            print(f"Watermark updated to: {new_watermark_str}")
            return
        except Exception as e:
            print(f"Failed to update watermark in {delete_config_table} (attempt {attempt}/{KUSTO_MAX_RETRIES}): {e}")
            if attempt >= KUSTO_MAX_RETRIES:
                raise RuntimeError(
                    f"Failed to update watermark in '{delete_config_table}' after {KUSTO_MAX_RETRIES} attempts. "
                    f"Aborting notebook."
                ) from e
            time.sleep(KUSTO_RETRY_BACKOFF_SECONDS * attempt)


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 91, Finished, Available, Finished, False)

In [ ]:
# --- Main Execution ---

# Initial run: lower_bound resolved from Lakehouse watermark inside get_alerts_to_process.
alerts_list = get_alerts_to_process(execution_start_time)
if alerts_list:
    results_to_log, exceptions_to_log = process_alerts(alerts_list)
    log_results(results_to_log, exceptions_to_log)

last_successful_upper_bound = execution_start_time

# --- Continuous Polling: run every minute for 1 hour ---
polling_duration_hours   = 1
polling_interval_seconds = 60

loop_start_time = datetime.now(timezone.utc)
loop_end_time   = loop_start_time + timedelta(hours=polling_duration_hours)

print(f"\nStarting continuous polling until {loop_end_time.isoformat()}")

iteration = 0

while True:
    now = datetime.now(timezone.utc)
    if now >= loop_end_time:
        print(f"\nPolling duration of {polling_duration_hours}h reached. Stopping.")
        break

    iteration += 1

    # The previous upper bound becomes the lower bound for this iteration.
    prev_upper_bound = last_successful_upper_bound
    current_upper_bound = datetime.now(timezone.utc)

    print(f"\n[Iteration {iteration}] {current_upper_bound.strftime('%H:%M:%S UTC')} | window: [{prev_upper_bound.strftime('%H:%M:%S')} - {current_upper_bound.strftime('%H:%M:%S')}]")

    alerts_list = get_alerts_to_process(current_upper_bound, lower_bound=prev_upper_bound)
    if alerts_list:
        results_to_log, exceptions_to_log = process_alerts(alerts_list)
        log_results(results_to_log, exceptions_to_log)

    last_successful_upper_bound = current_upper_bound

    if datetime.now(timezone.utc) < loop_end_time:
        print(f"Sleeping {polling_interval_seconds}s...")
        time.sleep(polling_interval_seconds)

# Update Lakehouse watermark once at the very end with the final upper bound.
update_watermark(last_successful_upper_bound)
print("Continuous polling finished.")


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 92, Submitted, Running, Running, True)

Successfully retrieved tokens.
Reading from Kusto query...
Found 0 alerts to process.
Updating DeleteLastRunTime watermark to 2026-04-16 12:35:29.121182+00:00...
Successfully updated watermark.

Starting continuous polling until 2026-04-16T13:36:05.498893+00:00

[Iteration 1] 12:36:05 UTC | window: [12:35:29 - 12:36:05]
Reading from Kusto query...
Found 0 alerts to process.
Sleeping 60s...

[Iteration 2] 12:37:05 UTC | window: [12:36:05 - 12:37:05]
Reading from Kusto query...
Found 0 alerts to process.
Sleeping 60s...

[Iteration 3] 12:38:05 UTC | window: [12:37:05 - 12:38:05]
Reading from Kusto query...
Found 0 alerts to process.
Sleeping 60s...

[Iteration 4] 12:39:06 UTC | window: [12:38:05 - 12:39:06]
Reading from Kusto query...
Found 0 alerts to process.
Sleeping 60s...

[Iteration 5] 12:40:06 UTC | window: [12:39:06 - 12:40:06]
Reading from Kusto query...
Found 0 alerts to process.
Sleeping 60s...

[Iteration 6] 12:41:06 UTC | window: [12:40:06 - 12:41:06]
Reading from Kusto quer